# 02. LoRA 学習 (SDXL / kohya-ss sd-scripts)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hiirocreate/personalizeAI/blob/main/notebooks/02_lora_training.ipynb)

自分のキャラクター・画風・人物を LoRA として学習します (Colab 無料 T4 で動作する省メモリ設定)。

1. 画像 15〜40 枚を用意 (③ 実行時にアップロード。保存先 `MyDrive/personalizeAI/datasets/<データ名>/` は自動作成)
2. 設定して上から実行 (目安: 20枚×10エポックで 30〜60 分)
3. 完成した LoRA は `MyDrive/personalizeAI/loras/` に保存 → `01_comfyui_server` の SDXL モードで使用

- イラスト/キャラ → ベース `animagine`、キャプション `wd14`
- 実写人物 → ベース `realvis`、キャプション `trigger_only` (例: トリガー `ohwx woman`)

⚠️ 実在人物の LoRA は本人の同意がある場合のみ。

In [ ]:
#@title ① 設定
USE_DRIVE = True  #@param {type:"boolean"}
DATASET = "mychara"  #@param {type:"string"}
#@markdown ✅ で ③ 実行時にスマホ/PC から画像 (または zip) をアップロード。Drive のフォルダは自動作成
UPLOAD = True  #@param {type:"boolean"}
TRIGGER = "mychara"  #@param {type:"string"}
BASE_MODEL = "animagine"  #@param ["animagine", "realvis"]
CAPTION = "wd14"  #@param ["wd14", "trigger_only", "existing"]
RESOLUTION = 1024  #@param [768, 896, 1024] {type:"raw"}
EPOCHS = 10  #@param {type:"integer"}
REPEATS = 10  #@param {type:"integer"}
NETWORK_DIM = 16  #@param [8, 16, 32] {type:"raw"}
LEARNING_RATE = 1e-4  #@param {type:"number"}
HF_TOKEN = ""  #@param {type:"string"}

In [ ]:
#@title ② インストール (初回 3〜5 分)

import os, subprocess
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode: print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError(cmd)
    return r.stdout

def dl(url, dst_dir, name=None):
    """aria2c で高速ダウンロード (既にあればスキップ)"""
    os.makedirs(dst_dir, exist_ok=True)
    name = name or url.split("/")[-1].split("?")[0]
    if os.path.exists(os.path.join(dst_dir, name)): return
    hdr = f'--header="Authorization: Bearer {HF_TOKEN}"' if HF_TOKEN and "huggingface.co" in url else ""
    print("↓", name); sh(f'aria2c -q -x16 -s16 -k1M --console-log-level=error {hdr} -d "{dst_dir}" -o "{name}" "{url}"')

if not os.path.exists("/usr/bin/aria2c"): sh("apt-get -qq install -y aria2")
if not os.path.exists("/content/personalizeAI"): sh("git clone -q --depth 1 https://github.com/hiirocreate/personalizeAI /content/personalizeAI")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/personalizeAI"
else:
    BASE = "/content/personalizeAI_data"
for d in ["loras", "outputs", "datasets", "3d"]: os.makedirs(f"{BASE}/{d}", exist_ok=True)
print("データ保存先:", BASE)

S = "/content/sd-scripts"
if not os.path.exists(S):
    sh(f"git clone -q --depth 1 https://github.com/kohya-ss/sd-scripts {S}")
    sh(f"cd {S} && pip install -q -r requirements.txt bitsandbytes onnxruntime-gpu")
CKPT = {
    "animagine": ("https://huggingface.co/cagliostrolab/animagine-xl-4.0/resolve/main/animagine-xl-4.0-opt.safetensors", "animagine-xl-4.0-opt.safetensors"),
    "realvis": ("https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors", "RealVisXL_V5.0_fp16.safetensors"),
}[BASE_MODEL]
dl(CKPT[0], "/content/models")
MODEL_PATH = f"/content/models/{CKPT[1]}"
print("✅ 準備完了")

In [ ]:
#@title ③ データセット準備 + キャプション生成
import glob, shutil
SRC = f"{BASE}/datasets/{DATASET}"
IMG = f"/content/train/{DATASET}"
shutil.rmtree(IMG, ignore_errors=True); os.makedirs(IMG)
os.makedirs(SRC, exist_ok=True)  # フォルダは自動作成
exts = (".png", ".jpg", ".jpeg", ".webp")
imgs = [f for f in glob.glob(f"{SRC}/*") if f.lower().endswith(exts)]
if UPLOAD or not imgs:
    from google.colab import files
    print(f"画像 (複数可) または zip を選択 → {SRC} に保存します")
    for name, data in files.upload().items():
        if name.lower().endswith(".zip"):
            import zipfile, io
            zipfile.ZipFile(io.BytesIO(data)).extractall(SRC)
        else:
            open(f"{SRC}/{name}", "wb").write(data)
    imgs = [f for f in glob.glob(f"{SRC}/**/*", recursive=True) if f.lower().endswith(exts) and "__MACOSX" not in f]
    for f in imgs:
        if os.path.dirname(f) != SRC: shutil.move(f, SRC)
    imgs = [f for f in glob.glob(f"{SRC}/*") if f.lower().endswith(exts)]
assert imgs, f"{SRC} に画像がありません"
for f in glob.glob(f"{SRC}/*"):
    if f.lower().endswith(exts + (".txt",)): shutil.copy(f, IMG)
print(len(imgs), "枚")
if CAPTION == "wd14":
    sh(f"cd {S} && python finetune/tag_images_by_wd14_tagger.py --onnx --repo_id SmilingWolf/wd-eva02-large-tagger-v3 "
       f"--batch_size 4 --caption_extension .txt --remove_underscore --thresh 0.35 {IMG}")
for f in imgs:
    txt = os.path.join(IMG, os.path.splitext(os.path.basename(f))[0] + ".txt")
    tags = open(txt).read().strip() if os.path.exists(txt) and CAPTION != "trigger_only" else ""
    with open(txt, "w") as w: w.write(", ".join(filter(None, [TRIGGER, tags])))
print("例:", open(txt).read()[:300])

In [ ]:
#@title ④ 学習 (T4 省メモリ設定: UNetのみ / fp16 / 8bit AdamW / gradient checkpointing)
OUT = f"{BASE}/loras"
open("/content/dataset.toml", "w").write(f"""
[general]
caption_extension = ".txt"
[[datasets]]
resolution = {RESOLUTION}
batch_size = 1
enable_bucket = true
[[datasets.subsets]]
image_dir = "{IMG}"
num_repeats = {REPEATS}
""")
cmd = f"""cd {S} && accelerate launch --num_processes 1 --num_machines 1 --mixed_precision fp16 --dynamo_backend no \
 sdxl_train_network.py --pretrained_model_name_or_path="{MODEL_PATH}" --dataset_config=/content/dataset.toml \
 --output_dir="{OUT}" --output_name="{DATASET}" --save_model_as=safetensors \
 --network_module=networks.lora --network_dim={NETWORK_DIM} --network_alpha={NETWORK_DIM // 2} --network_train_unet_only \
 --learning_rate={LEARNING_RATE} --optimizer_type=AdamW8bit --lr_scheduler=cosine --lr_warmup_steps=50 \
 --max_train_epochs={EPOCHS} --save_every_n_epochs=2 --mixed_precision=fp16 --save_precision=fp16 \
 --cache_latents --cache_latents_to_disk --cache_text_encoder_outputs --gradient_checkpointing --sdpa \
 --no_half_vae --max_data_loader_n_workers=1 --seed=42"""
!{cmd}
print("✅ 保存先:", OUT); print(os.listdir(OUT))